In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting LLMs with validation set, ask to find hallucinations and label them.
## Model used: gemini-1.5-flash
## required files: llabel_validation_set_with_llm_for_baseline.env
##                 mushroom.en-val.v2.unlabeled.jsonl

In [ ]:
#INSTALL DEPENDENCIES

!pip install google-generativeai

In [ ]:
# IMPORT LIBRARIES

import configparser
import google.generativeai as genai
import json

from google.colab import userdata

import random

from google.colab import drive
drive.mount('/content/drive')

import time
import textwrap
import re

Mounted at /content/drive


In [ ]:
# DEFINE VARIABLES

Your_API_Key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=Your_API_Key)

model_used = "gemini-1.5-flash"
model = genai.GenerativeModel(model_name="gemini-1.5-flash")
prompts = configparser.ConfigParser()

prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline.env')

#output_file="/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini1.5.jsonl"
output_file="/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_ChatGPT4o.jsonl"

set_to_label =  '/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/mushroom.en-val.v2.unlabeled.jsonl'

In [ ]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini1.5.jsonl is created or already exists.


In [ ]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(set_to_label)


In [ ]:
#FUNCTIONS

In [ ]:
# COMPILE PROMPTS
def define_prompt(datapoint):

  #samples can be also imported from a jsonl file.
  Sample1 = prompts.get('SAMPLES', 'Sample1')

  prompt1 = (
      f"{prompts.get('PROMPTS', 'p0')}"
      f"{datapoint}"
      f"{prompts.get('PROMPTS', 'p1')}"
      f"{prompts.get('PROMPTS', 'p2')}"
      f"{prompts.get('PROMPTS', 'p3')}"
      f"{prompts.get('PROMPTS', 'p4')}"
      f"{prompts.get('PROMPTS', 'p5')}"
      )

  return prompt1

In [ ]:
## TESTING define_prompt()
d = data[30]
print(d)
print()
print(textwrap.fill(define_prompt(d), width=100))

{'id': 'val-en-31_unlabeled', 'lang': 'EN', 'model_input': 'How big is a western oriole?', 'model_output_text': ' A Western Oriole is a medium-sized bird, with an average length of 15.3 cm (6 inches) and a wingspan of around 20.3 cm (8 inches). They weigh between 19-34 grams (0.7-1.2 ounces). Their bill is long and curved, typically measuring about 3.5 cm (1.4 inches) in length.', 'model_id': 'TheBloke/Mistral-7B-Instruct-v0.2-GGUF', 'soft_labels': [], 'hard_labels': [], 'model_output_logits': [-2.2387826443, 0.4497404695, 1.8324928284, 5.1005496979, -2.2169632912, 3.3091645241, 4.1546559334, 2.4361951351, -1.7116131783, 2.3883566856, 5.1600074768, 6.4319458008, 4.1633906364, 6.4638123512, 5.9917187691, 2.1523523331, 7.2986679077, 1.0490045547, 23.5293197632, 1.5084402561, 8.2483158112, 4.0005364418, 4.5845832825, -0.2383529544, -1.755369544, 3.2650258541, 1.1170085669, 7.0609579086, 2.7663283348, 4.2022938728, 1.1904013157, 6.5613508224, 5.3330936432, 2.805464983, 24.8552436829, 2.346

In [ ]:
## Testing
model.generate_content(define_prompt(d)).text

'15.3 cm (6 inches)\n(17, 32)\n20.3 cm (8 inches)\n(34, 49)\n19-34 grams (0.7-1.2 ounces)\n(51, 73)\n3.5 cm (1.4 inches)\n(75, 90)\n\n'

In [ ]:
## Testing
h = model.generate_content(define_prompt(d)).text.split("\n")
print(h)

['15.3 cm (6 inches)', '(17,32)', '20.3 cm (8 inches)', '(40,55)', '19-34 grams (0.7-1.2 ounces)', '(58,80)', '3.5 cm (1.4 inches)', '(88,103)', '', '']


In [ ]:
## do not use this function separately, since every call of model.generate_content produces slightly different output
def process_data(datapointX):
    hallucinated_words = [list_element for list_element in model.generate_content(define_prompt(datapointX)).text.split("\n") if list_element] ## this ensure list_element is not empty
    return hallucinated_words

In [ ]:
## TESTING process_data() function
h = process_data(d)
print(h)

for element in h:
  print(element)

['15.3 cm (6 inches)', '(17, 34)', '20.3 cm (8 inches)', '(41, 58)', '3.5 cm (1.4 inches)', '(102, 119)']
15.3 cm (6 inches)
(17, 34)
20.3 cm (8 inches)
(41, 58)
3.5 cm (1.4 inches)
(102, 119)


In [ ]:
def extract_spans(datapointX):
    hallucinated_words = process_data(datapointX)
    #print(hallucinated_words)
    spans = []
    for id, element in enumerate(hallucinated_words):
      if id%2 == 1:
        span = list(int(x) for x in element.strip("'()",).split(","))
        #print(span)
        spans.append(span)
    return hallucinated_words, spans

In [ ]:
## Testing Extract spans
print(d["model_output_text"])
extract_spans(d)

 A Western Oriole is a medium-sized bird, with an average length of 15.3 cm (6 inches) and a wingspan of around 20.3 cm (8 inches). They weigh between 19-34 grams (0.7-1.2 ounces). Their bill is long and curved, typically measuring about 3.5 cm (1.4 inches) in length.


(['15.3 cm (6 inches)',
  '(17, 34)',
  '20.3 cm (8 inches)',
  '(40, 57)',
  '3.5 cm (1.4 inches)',
  '(108, 125)'],
 [[17, 34], [40, 57], [108, 125]])

In [ ]:
def last_line_number(file_path):
  # Read the last line of the file
  last_line = None
  with open(file_path, "r") as file:
      for line in file:
          last_line = line.strip()  # Store the current line

  # Parse the JSON object from the last line
  if last_line:
      last_data = json.loads(last_line)
      #print("Last JSON object:", last_data)
      return last_data["number"]
  else:
      print("The file is empty")
      return None


In [ ]:
##TESTING
last_line_number(output_file)

The file is empty


In [ ]:
for key, value in data[0].items():
  print(f'"{key}":') #type(value))

"id":
"lang":
"model_input":
"model_output_text":
"model_id":
"soft_labels":
"hard_labels":
"model_output_logits":
"model_output_tokens":


In [ ]:
def label_and_save_data(data):
    processed_count = 0  # Counter for newly processed entries

    for number, datapoint in enumerate(data, start=1):
        # Get the last processed ID from the output file
        last_processed_number = last_line_number(output_file) or 0
        #print("Last processed ID:", last_processed_number)

        # Skip already processed entries
        if number <= last_processed_number:
            continue

        # Define prompt and process data
        prompt = define_prompt(datapoint)
        hallucinated_words, hard_labels = extract_spans(datapoint)

        # Save the datapoint to the JSONL file
        with open(output_file, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "number": number,
                "id": datapoint["id"],
                "lang": datapoint["lang"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_id": datapoint["model_id"],
                "hallucinated_words": hallucinated_words,
                "soft_labels": [],
                "hard_labels": hard_labels,
                "model_output_logits": datapoint["model_output_logits"],
                "model_output_tokens": datapoint["model_output_tokens"],
            }

            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")
        print(datapoint["model_output_text"])
        print(hallucinated_words)
        print(hard_labels)
        print("-------------------------------")

        # Increment the processed count
        processed_count += 1

        # Stop processing after 5 new entries, wait for 10 seconds
        if processed_count >= 5:
            print(f"Processed {processed_count} entries. Waiting for 10 sec. at ID {number}.")
            time.sleep(10)

            processed_count = 0


In [ ]:
label_and_save_data(data)

 No, Chris Claremont did not write the entire "Days of Future Past" story on his own. The original X-Men comic book storyline, which was published in issues #141-142 of Uncanny X-Men in 1981, was written by Chris Claremont and illustrated by John Byrne. However, the alternative future sequences were drawn by Terry Austin, and other artists contributed to the later adaptations of this story into other media formats like animation and film. So while Claremont's role is significant, it's important to acknowledge the contributions of other creators involved in bringing "Days of Future Past" to life.
['other artists contributed to the later adaptations of this story into other media formats like animation and film', '(153, 228)']
[[153, 228]]
-------------------------------
 Toruń, also known as Thorn in German, was a member of the Hanseatic League from approximately 1260 until its expulsion in 1524. The Hanseatic League was a commercial and defensive confederation of merchant guilds and ma

In [ ]:
output_file

'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gemini1.5.jsonl'

In [ ]:
## Resave a copy with no extra keys.

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')

        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels']]
        print(soft_labels)

        # Save the datapoint to the JSONL file
        with open("mushroom.en-val.v2.unlabeled.labelled_with_gemini1.5_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")



[[16, 22], [23, 58]]
[{'start': 16, 'prob': 1.0, 'end': 22}, {'start': 23, 'prob': 1.0, 'end': 58}]
[[0, 9], [9, 10], [10, 15]]
[{'start': 0, 'prob': 1.0, 'end': 9}, {'start': 9, 'prob': 1.0, 'end': 10}, {'start': 10, 'prob': 1.0, 'end': 15}]
[[10, 18], [20, 28]]
[{'start': 10, 'prob': 1.0, 'end': 18}, {'start': 20, 'prob': 1.0, 'end': 28}]
[[0, 4]]
[{'start': 0, 'prob': 1.0, 'end': 4}]
[[19, 102], [103, 104]]
[{'start': 19, 'prob': 1.0, 'end': 102}, {'start': 103, 'prob': 1.0, 'end': 104}]
[[208, 255], [157, 191], [104, 139], [70, 103]]
[{'start': 208, 'prob': 1.0, 'end': 255}, {'start': 157, 'prob': 1.0, 'end': 191}, {'start': 104, 'prob': 1.0, 'end': 139}, {'start': 70, 'prob': 1.0, 'end': 103}]
[[29, 40], [52, 55]]
[{'start': 29, 'prob': 1.0, 'end': 40}, {'start': 52, 'prob': 1.0, 'end': 55}]
[[0, 7]]
[{'start': 0, 'prob': 1.0, 'end': 7}]
[[41, 55]]
[{'start': 41, 'prob': 1.0, 'end': 55}]
[[12, 19], [20, 23]]
[{'start': 12, 'prob': 1.0, 'end': 19}, {'start': 20, 'prob': 1.0, 'end':